In [2]:
import numpy as np
import cv2
import skimage.feature as sk
#import matplotlib.pyplot as plt

In [3]:
# Load the 2 images
left_img = cv2.imread('left_im.png')
center_img = cv2.imread('center_im.png')
right_img = cv2.imread('right_im.png')

images = [left_img, center_img, right_img]

In [4]:
left_img.shape

(774, 414, 3)

In [5]:
# harrys corner detection
def corner_harris(image, block_size, ksize, k):
    # Convert image to gray scale
    gray_im = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    gray_im = np.float32(gray_im)

    # Compute the corner response function for each pixel in the image
    features_im = cv2.cornerHarris(gray_im, block_size, ksize, k)

    # Dilate the corner response function to enhance the corner points
    features_im = cv2.dilate(features_im, None)

    # Need create a new instance of the image, otherwise we cant run the test with different parameters
    # Has it will modify the original image and the next test will be run on the modified image
    image_copy = image.copy()

    # Mark the corners in the original image with a red pixel
    image_copy[features_im > 0.01 * features_im.max()] = [255, 0, 0]
    harris_features = features_im > 0.01 * features_im.max()

    return image_copy, harris_features

# Investigating the effect of the parameters on the size and number of detected corners
"""
test1_1_left, _ = corner_harris(left_img, 17, 21, 0.01)
test1_2_left, _ = corner_harris(left_img, 9, 13, 0.01)
test1_3_left, _ = corner_harris(left_img, 5, 7, 0.01)
test1_4_left, _ = corner_harris(left_img, 3, 5, 0.01)

cv2.imshow('test1_1_left', test1_1_left)
cv2.imshow('test1_2_left', test1_2_left)
cv2.imshow('test1_3_left', test1_3_left)
cv2.imshow('test1_4_left', test1_4_left)
cv2.waitKey(0)
cv2.destroyAllWindows()

test2_1_left, _ = corner_harris(left_img, 5, 7, 0.01)
test2_2_left, _ = corner_harris(left_img, 5, 7, 0.04)
test2_3_left, _ = corner_harris(left_img, 5, 7, 0.06)
test2_4_left, _ = corner_harris(left_img, 5, 7, 0.1)

cv2.imshow('test2_1_left', test2_1_left)
cv2.imshow('test2_2_left', test2_2_left)
cv2.imshow('test2_3_left', test2_3_left)
cv2.imshow('test2_4_left', test2_4_left)
cv2.waitKey(0)
cv2.destroyAllWindows()
"""


"\ntest1_1_left, _ = corner_harris(left_img, 17, 21, 0.01)\ntest1_2_left, _ = corner_harris(left_img, 9, 13, 0.01)\ntest1_3_left, _ = corner_harris(left_img, 5, 7, 0.01)\ntest1_4_left, _ = corner_harris(left_img, 3, 5, 0.01)\n\ncv2.imshow('test1_1_left', test1_1_left)\ncv2.imshow('test1_2_left', test1_2_left)\ncv2.imshow('test1_3_left', test1_3_left)\ncv2.imshow('test1_4_left', test1_4_left)\ncv2.waitKey(0)\ncv2.destroyAllWindows()\n\ntest2_1_left, _ = corner_harris(left_img, 5, 7, 0.01)\ntest2_2_left, _ = corner_harris(left_img, 5, 7, 0.04)\ntest2_3_left, _ = corner_harris(left_img, 5, 7, 0.06)\ntest2_4_left, _ = corner_harris(left_img, 5, 7, 0.1)\n\ncv2.imshow('test2_1_left', test2_1_left)\ncv2.imshow('test2_2_left', test2_2_left)\ncv2.imshow('test2_3_left', test2_3_left)\ncv2.imshow('test2_4_left', test2_4_left)\ncv2.waitKey(0)\ncv2.destroyAllWindows()\n"

In [6]:
harris_output = [corner_harris(img, 5, 7, 0.06) for img in images] 
harris_RGB_image = [cv2.cvtColor(x[0], cv2.COLOR_BGR2RGB) for x in harris_output]


In [7]:
def compute_keyPoints(harris_features): 
    keypoints = []
    for i in range(harris_features.shape[0]):
        for j in range(harris_features.shape[1]):
            if harris_features[i, j]:
                keypoints.append(cv2.KeyPoint(j, i, size=16))
    return keypoints

In [22]:
# Compute SIFT features
# First exctract the key points in each image
key_points = [compute_keyPoints(x[1]) for x in harris_output]

# We have way to many points from our Harris corner detection,
print(len(key_points[0]))

best_kps = []
for keypoints in key_points:
    best_kp = sorted(keypoints, key=lambda kp: kp.response, reverse=True)
    new_kp = best_kp[:1500]

    best_kps.append(new_kp)

print(len(best_kps[0]))


20412
1500


In [29]:
# Then compute the SIFT features for each image using the key points
sift = cv2.xfeatures2d.SIFT_create()
SIFT_features = [sift.compute(img, kp) for img, kp in zip(images, best_kps)]
print(SIFT_features[0])


((< cv2.KeyPoint 000001AE2B70F600>, < cv2.KeyPoint 000001AE2B70F570>, < cv2.KeyPoint 000001AE2B70F7B0>, < cv2.KeyPoint 000001AE2B70ED00>, < cv2.KeyPoint 000001AE2B70F6C0>, < cv2.KeyPoint 000001AE2B70FD80>, < cv2.KeyPoint 000001AE2B70ED90>, < cv2.KeyPoint 000001AE2B70ECD0>, < cv2.KeyPoint 000001AE2B70F390>, < cv2.KeyPoint 000001AE2B70E820>, < cv2.KeyPoint 000001AE2B70E8E0>, < cv2.KeyPoint 000001AE2B70F8D0>, < cv2.KeyPoint 000001AE2B70F630>, < cv2.KeyPoint 000001AE2B70F7E0>, < cv2.KeyPoint 000001AE2B70F330>, < cv2.KeyPoint 000001AE2B70EA60>, < cv2.KeyPoint 000001AE2B70F870>, < cv2.KeyPoint 000001AE2B70EAC0>, < cv2.KeyPoint 000001AE2B70EA30>, < cv2.KeyPoint 000001AE2B70FD20>, < cv2.KeyPoint 000001AE2B70EE50>, < cv2.KeyPoint 000001AE2878BC90>, < cv2.KeyPoint 000001AE2848CF00>, < cv2.KeyPoint 000001AE2848CDE0>, < cv2.KeyPoint 000001AE2848D1A0>, < cv2.KeyPoint 000001AE2848CDB0>, < cv2.KeyPoint 000001AE2848CC60>, < cv2.KeyPoint 000001AE2848D020>, < cv2.KeyPoint 000001AE2848CFC0>, < cv2.KeyPoi

In [ ]:
#Compute the distances between every descriptor in image 1 with every descriptor in image 2. For this, use: 
# a) Normalized correlation 
# b) Euclidean distance after normalizing each descriptor.

def normalize_descriptors(descriptors):
    norms = np.linalg.norm(descriptors, axis=1, keepdims=True)
    return descriptors / norms


def euclidean_distance(desc1, desc2):
    return np.linalg.norm(desc1[:, None, :] - desc2[None, :, :], axis=2)


def normalized_correlation(desc1, desc2):
    return np.dot(desc1, desc2.T)

normalized_SIFT_features = [normalize_descriptors(x[1]) for x in SIFT_features]

euclidean_distances = euclidean_distance(normalized_SIFT_features[1], normalized_SIFT_features[2])
correlation = normalized_correlation(normalized_SIFT_features[1], normalized_SIFT_features[2])

print("Euclidean Distances:\n", euclidean_distances)
print("\nNormalized Correlations:\n", correlation)

Euclidean Distances:
 [[0.69659996 0.6838089  0.66989654 ... 0.7584817  0.7593931  0.7602415 ]
 [0.70072496 0.6878897  0.6739092  ... 0.7585141  0.7593702  0.76014334]
 [0.7062652  0.69341445 0.6793773  ... 0.75835747 0.7591785  0.75988674]
 ...
 [0.8136253  0.80324566 0.7942401  ... 0.35113826 0.34609652 0.3404047 ]
 [0.81332827 0.8029018  0.79385006 ... 0.34805664 0.34296015 0.33718315]
 [0.81356114 0.8031045  0.79401666 ... 0.34704018 0.34184295 0.33588058]]

Normalized Correlations:
 [[0.75737417 0.76620257 0.7756193  ... 0.7123529  0.7116609  0.7110163 ]
 [0.7544923  0.76340395 0.7729231  ... 0.7123282  0.7116787  0.711091  ]
 [0.75059474 0.7595882  0.7692233  ... 0.71244705 0.7118239  0.7112862 ]
 ...
 [0.6690069  0.6773981  0.68459135 ... 0.9383509  0.94010866 0.94206226]
 [0.6692486  0.67767435 0.684901   ... 0.9394282  0.9411893  0.9431537 ]
 [0.6690591  0.67751163 0.68476874 ... 0.93978167 0.94157165 0.9435921 ]]


In [59]:
distances = []
correlations = []
for i in range(len(normalized_SIFT_features)):
    for j in range(i + 1, len(normalized_SIFT_features)):
        euclidean_distances = euclidean_distance(normalized_SIFT_features[i], normalized_SIFT_features[j])
        correlation = normalized_correlation(normalized_SIFT_features[i], normalized_SIFT_features[j])
        distances.append(euclidean_distances)
        correlations.append(correlation)

In [61]:
#Now that we have the distances and correlations, we can find the best matches between the descriptors 
def find_matches(correlations, threshold):
    matches = []
    for i in range(correlations.shape[0]):
        for j in range(correlations.shape[1]):
            if correlations[i, j] > threshold:
                matches.append((i, j))

    return matches

# Lets find the best value for the threshold: ]
for corr in correlations:
    for threshold in [ 0.7, 0.75, 0.8, 0.85, 0.86, 0.87, 0.88, 0.89, 0.895, 0.9]:
        matches = find_matches(corr, threshold)
        print(f"Threshold: {threshold}, Number of Matches: {len(matches)}")


Threshold: 0.7, Number of Matches: 1243349
Threshold: 0.75, Number of Matches: 668500
Threshold: 0.8, Number of Matches: 172074
Threshold: 0.85, Number of Matches: 48124
Threshold: 0.86, Number of Matches: 35425
Threshold: 0.87, Number of Matches: 23812
Threshold: 0.88, Number of Matches: 13050
Threshold: 0.89, Number of Matches: 4329
Threshold: 0.895, Number of Matches: 1816
Threshold: 0.9, Number of Matches: 316
Threshold: 0.7, Number of Matches: 997101
Threshold: 0.75, Number of Matches: 451380
Threshold: 0.8, Number of Matches: 97203
Threshold: 0.85, Number of Matches: 13460
Threshold: 0.86, Number of Matches: 7125
Threshold: 0.87, Number of Matches: 2544
Threshold: 0.88, Number of Matches: 560
Threshold: 0.89, Number of Matches: 151
Threshold: 0.895, Number of Matches: 49
Threshold: 0.9, Number of Matches: 3
Threshold: 0.7, Number of Matches: 2152613
Threshold: 0.75, Number of Matches: 1961135
Threshold: 0.8, Number of Matches: 1612588
Threshold: 0.85, Number of Matches: 1167393
T

In [75]:
#We can see a big variance in the number of matches for the different image pairs, so I will use a top-k approach instead
# TODO: plot match quality vs k just as you would vs threshold.

def find_top_k_matches(correlations, max, k=200):
    matches = []
    # Find the cut off point, when using correlation we want the max value while for distances we want the smallest ones
    if max:
        cut_off = np.sort(correlations.flatten())[-k-1]
    else:
        cut_off = np.sort(correlations.flatten())[k-1]

    # Find the matches based on the cutoff point
    for i in range(correlations.shape[0]):
        for j in range(correlations.shape[1]):
            if correlations[i, j] > cut_off:
                matches.append((i, j))

    return matches

filtered_corr_matches = [find_top_k_matches(corr, max=True) for corr in correlations]
filtered_dist_matches = [find_top_k_matches(dist, max=False) for dist in distances]    